# Bab 12: Ensemble: Bagging, Random Forest, dan Boosting

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab12/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab12")
print("Folder kerja:", os.getcwd())

## Listing 12.1: Satu pohon, bagging, dan random forest pada data churn

`kode/bab12/bagging.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab12')
import numpy as np
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from siapkan import muat, pra_pohon

X, y, num = muat()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
rng = np.random.default_rng(0)
idx = rng.integers(0, len(Xl), len(Xl))
print(f"satu sampel bootstrap: {len(np.unique(idx)) / len(Xl):.1%} "
      "baris unik (teori 63,2%)")

skor = {}
for nama, m in {
        "satu pohon penuh": DecisionTreeClassifier(random_state=0),
        "bagging 200 pohon": BaggingClassifier(
            DecisionTreeClassifier(), n_estimators=200,
            oob_score=True, random_state=0, n_jobs=-1),
        "random forest 200": RandomForestClassifier(
            200, max_features="sqrt", oob_score=True,
            random_state=0, n_jobs=-1)}.items():
    pipa = make_pipeline(pra_pohon(num), m).fit(Xl, yl)
    p = pipa.predict_proba(Xu)[:, 1]
    oob = getattr(m, "oob_score_", float("nan"))
    print(f"{nama:18s} AUC uji {roc_auc_score(yu, p):.3f}  "
          f"akurasi OOB {oob:.3f}")
    skor[nama] = p

print("\nkorelasi prediksi antarpohon (rata-rata, 30 pohon pertama):")
for nama in ["bagging 200 pohon", "random forest 200"]:
    m = (BaggingClassifier(DecisionTreeClassifier(), n_estimators=30,
                           random_state=0)
         if nama.startswith("bagging") else
         RandomForestClassifier(30, max_features="sqrt", random_state=0))
    pipa = make_pipeline(pra_pohon(num), m).fit(Xl, yl)
    Z = pipa[0].transform(Xu)
    P = np.array([t.predict_proba(np.asarray(Z))[:, 1]
                  for t in m.estimators_])
    r = np.corrcoef(P)[np.triu_indices(30, 1)].mean()
    print(f"  {nama:18s} rho = {r:.3f}")

## Listing 12.2: Pengaruh jumlah pohon dan max_features

`kode/bab12/jumlah_pohon.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab12')
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from siapkan import muat, pra_pohon

X, y, num = muat()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
pra = pra_pohon(num).fit(Xl)
Zl, Zu = pra.transform(Xl), pra.transform(Xu)
hasil = []
print("AUC uji setelah B pohon; B =  1      5      20     100    500")
for mf in ["sqrt", 0.5, None]:
    rf = RandomForestClassifier(500, max_features=mf, oob_score=True,
                                min_samples_leaf=3, random_state=0,
                                n_jobs=-1).fit(Zl, yl)
    P = np.array([t.predict_proba(Zu)[:, 1] for t in rf.estimators_])
    kum = np.cumsum(P, axis=0) / np.arange(1, 501)[:, None]
    auc = [roc_auc_score(yu, kum[b - 1]) for b in (1, 5, 20, 100, 500)]
    hasil.append([roc_auc_score(yu, kum[b - 1]) for b in range(1, 501)])
    print(f"max_features={mf!s:5s}: " + "  ".join(f"{a:.3f}" for a in auc))
with open("gambar/bab12/jumlah_pohon.dat", "w") as f:
    f.write("b sqrt setengah semua\n")
    for b in list(range(1, 20)) + list(range(20, 501, 10)):
        f.write(f"{b} " + " ".join(f"{h[b - 1]:.4f}" for h in hasil)
                + "\n")

## Listing 12.3: Tiga putaran AdaBoost pada delapan titik

`kode/bab12/adaboost_mini.py`

In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4, 5, 6, 7, 8])
y = np.array([1, 1, -1, -1, 1, 1, 1, -1])
w = np.full(8, 1 / 8)


def tunggul_terbaik(x, y, w):
    terbaik = None
    for t in np.arange(1.5, 8, 1.0):
        for arah in [1, -1]:
            h = np.where(x < t, arah, -arah)
            e = w[h != y].sum()
            if terbaik is None or e < terbaik[0] - 1e-12:
                terbaik = (e, t, arah, h)
    return terbaik


F = np.zeros(8)
for m in range(1, 4):
    e, t, arah, h = tunggul_terbaik(x, y, w)
    alfa = 0.5 * np.log((1 - e) / e)
    print(f"putaran {m}: h(x) = {arah:+d} jika x < {t}, "
          f"selain itu {-arah:+d}")
    print(f"  galat berbobot e = {e:.4f}, alfa = {alfa:.4f}")
    w = w * np.exp(-alfa * y * h)
    w = w / w.sum()
    print("  bobot baru:", np.round(w, 4))
    F += alfa * h
    print("  skor F:", np.round(F, 3), "-> salah",
          int((np.sign(F) != y).sum()))

## Listing 12.4: Gradient boosting dengan tangan untuk belanja bulan depan

`kode/bab12/gb_manual.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab12')
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from siapkan import DATA, muat, pra_pohon

X, _, num = muat()
y = pd.read_csv(DATA + "pelanggan.csv")["belanja_bulan_depan"] / 1000
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, random_state=7)
pra = pra_pohon(num).fit(Xl)
Zl, Zu = pra.transform(Xl), pra.transform(Xu)

nu, M = 0.1, 300
Fl = np.full(len(yl), yl.mean())
Fu = np.full(len(yu), yl.mean())
for m in range(1, M + 1):
    r = yl.to_numpy() - Fl
    h = DecisionTreeRegressor(max_depth=3, random_state=0).fit(Zl, r)
    Fl += nu * h.predict(Zl)
    Fu += nu * h.predict(Zu)
    if m in (1, 10, 50, 100, 300):
        print(f"M={m:3d}: MAE latih {mean_absolute_error(yl, Fl):6.1f}"
              f"  MAE uji {mean_absolute_error(yu, Fu):6.1f}")

sk = GradientBoostingRegressor(n_estimators=M, learning_rate=nu,
                               max_depth=3, random_state=0).fit(Zl, yl)
print(f"sklearn GradientBoostingRegressor: MAE uji "
      f"{mean_absolute_error(yu, sk.predict(Zu)):.1f}")

## Listing 12.5: Laju belajar dan jumlah pohon pada gradient boosting

`kode/bab12/laju_belajar.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab12')
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from siapkan import muat, pra_pohon

X, y, num = muat()
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
pra = pra_pohon(num).fit(Xl)
Zl, Zu = pra.transform(Xl), pra.transform(Xu)
kurva = {}
for nu in [1.0, 0.3, 0.05]:
    gb = GradientBoostingClassifier(n_estimators=600, learning_rate=nu,
                                    max_depth=3, subsample=0.8,
                                    random_state=0).fit(Zl, yl)
    ll = [log_loss(yu, p[:, 1]) for p in gb.staged_predict_proba(Zu)]
    kurva[nu] = ll
    b = int(np.argmin(ll))
    print(f"laju {nu:4.2f}: log loss uji minimum {ll[b]:.4f} pada "
          f"{b + 1:3d} pohon; setelah 600 pohon {ll[-1]:.4f}")
with open("gambar/bab12/laju.dat", "w") as f:
    f.write("m nu1 nu03 nu005\n")
    for m in list(range(1, 50)) + list(range(50, 601, 10)):
        f.write(f"{m} " + " ".join(f"{kurva[nu][m - 1]:.4f}"
                                   for nu in [1.0, 0.3, 0.05]) + "\n")

## Listing 12.6: XGBoost, LightGBM, dan HistGradientBoosting dengan penghentian dini

`kode/bab12/xgb_lgbm.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab12')
import time
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from siapkan import muat

X, y, num = muat()
X = X.astype({k: "category" for k in ["kota", "tingkat",
                                      "jenis_kelamin"]})
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
Xa, Xv, ya, yv = train_test_split(Xl, yl, test_size=0.2, stratify=yl,
                                  random_state=1)
model = {
    "XGBoost": xgb.XGBClassifier(
        n_estimators=2000, learning_rate=0.03, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        enable_categorical=True, tree_method="hist",
        early_stopping_rounds=100, eval_metric="logloss",
        random_state=0),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=15,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        min_child_samples=40, random_state=0, verbose=-1),
    "HistGB sklearn": HistGradientBoostingClassifier(
        max_iter=2000, learning_rate=0.03, max_leaf_nodes=15,
        categorical_features="from_dtype", early_stopping=True,
        validation_fraction=0.2, n_iter_no_change=100,
        random_state=0)}
for nama, m in model.items():
    t0 = time.perf_counter()
    if nama == "XGBoost":
        m.fit(Xa, ya, eval_set=[(Xv, yv)], verbose=False)
        n = m.best_iteration + 1
    elif nama == "LightGBM":
        m.fit(Xa, ya, eval_set=[(Xv, yv)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        n = m.best_iteration_
    else:
        m.fit(Xl, yl)
        n = m.n_iter_
    dt = time.perf_counter() - t0
    auc = roc_auc_score(yu, m.predict_proba(Xu)[:, 1])
    print(f"{nama:15s} AUC uji {auc:.3f}  pohon {n:4d}  "
          f"waktu {dt:4.1f} dtk")

## Listing 12.7: Pentingnya impuritas, pentingnya permutasi, dan ketergantungan parsial

`kode/bab12/pentingnya.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab12')
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import partial_dependence, permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from siapkan import muat, pra_pohon

X, y, num = muat()
rng = np.random.default_rng(0)
X = X.assign(acak=rng.normal(size=len(X)),
             id_acak=rng.integers(0, 10_000, len(X)))
num = num + ["acak", "id_acak"]
X[num] = X[num].astype(float)
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
rf = make_pipeline(pra_pohon(num), RandomForestClassifier(
    300, random_state=0, n_jobs=-1)).fit(Xl, yl)
nama = rf[0].get_feature_names_out()
mdi = pd.Series(rf[-1].feature_importances_, index=nama)
perm = permutation_importance(rf, Xu, yu, scoring="roc_auc",
                              n_repeats=10, random_state=0, n_jobs=-1)
pi = pd.Series(perm.importances_mean, index=X.columns)
tab = pd.DataFrame({"MDI": mdi, "permutasi": pi}).sort_values(
    "permutasi", ascending=False)
print(tab.head(8).round(4).to_string())
print("...")
print(tab.loc[["acak", "id_acak"]].round(4).to_string())

pd_ = partial_dependence(rf, Xl, ["recency_hari"],
                         grid_resolution=30)
with open("gambar/bab12/pdp.dat", "w") as f:
    f.write("recency p\n")
    for r, p in zip(pd_["grid_values"][0], pd_["average"][0]):
        f.write(f"{r:.1f} {p:.4f}\n")
print("PDP recency: p churn rata-rata naik dari "
      f"{pd_['average'][0][0]:.3f} ke {pd_['average'][0][-1]:.3f}")

## Listing 12.8: Studi kasus: logistik, random forest, LightGBM, dan stacking

`kode/bab12/studi_kasus.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab12')
import lightgbm as lgb
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             roc_auc_score)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from siapkan import KATEGORIK, muat, pra_pohon

X, y, num = muat(tren=True)
Xl, Xu, yl, yu = train_test_split(X, y, test_size=0.3, stratify=y,
                                  random_state=7)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
logistik = make_pipeline(ColumnTransformer(
    [("num", StandardScaler(), num),
     ("kat", OneHotEncoder(handle_unknown="ignore"), KATEGORIK)]),
    LogisticRegression(C=3.162, max_iter=3000))
hutan = make_pipeline(pra_pohon(num), RandomForestClassifier(
    500, min_samples_leaf=5, max_features=0.3, random_state=0,
    n_jobs=-1))
ruang = {"lgbmclassifier__num_leaves": [4, 7, 15, 31],
         "lgbmclassifier__learning_rate": [0.01, 0.02, 0.05],
         "lgbmclassifier__n_estimators": [200, 400, 800],
         "lgbmclassifier__min_child_samples": [20, 50, 100],
         "lgbmclassifier__colsample_bytree": [0.5, 0.8, 1.0],
         "lgbmclassifier__reg_lambda": [0, 1, 10]}
cari = RandomizedSearchCV(make_pipeline(pra_pohon(num), lgb.LGBMClassifier(
    subsample=0.8, subsample_freq=1, random_state=0, verbose=-1,
    n_jobs=1)),
    ruang, n_iter=15, cv=cv, scoring="roc_auc", random_state=0,
    n_jobs=-1).fit(Xl, yl)
gbm = cari.best_estimator_
print("LightGBM terbaik:", {k.split("__")[1]: v
                            for k, v in cari.best_params_.items()})
tumpuk = StackingClassifier(
    [("log", logistik), ("gbm", gbm)],
    final_estimator=LogisticRegression(), cv=cv, n_jobs=-1)

print("\nmodel           AUC-CV  AUC-uji  AP-uji  Brier   laba(jt)")
p_uji = {}
for nama, m in [("logistik", logistik), ("random forest", hutan),
                ("LightGBM", gbm), ("tumpuk", tumpuk)]:
    auc_cv = cross_val_score(m, Xl, yl, cv=cv, scoring="roc_auc",
                             n_jobs=-1).mean()
    p = m.fit(Xl, yl).predict_proba(Xu)[:, 1]
    p_uji[nama] = p
    sasar = p >= 15 / 54
    laba = (54_000 * yu[sasar].sum() - 15_000 * sasar.sum()) / 1e6
    print(f"{nama:14s} {auc_cv:7.3f} {roc_auc_score(yu, p):8.3f} "
          f"{average_precision_score(yu, p):7.3f} "
          f"{brier_score_loss(yu, p):7.4f} {laba:8.1f}")

rng = np.random.default_rng(0)
yv = yu.to_numpy()
beda = []
for _ in range(1000):
    i = rng.integers(0, len(yv), len(yv))
    beda.append(roc_auc_score(yv[i], p_uji["tumpuk"][i])
                - roc_auc_score(yv[i], p_uji["logistik"][i]))
lo, hi = np.percentile(beda, [2.5, 97.5])
print(f"\nselisih AUC tumpuk - logistik: {np.mean(beda):+.4f} "
      f"[{lo:+.4f}, {hi:+.4f}]")
r = np.corrcoef(p_uji["logistik"], p_uji["LightGBM"])[0, 1]
print(f"korelasi skor logistik vs LightGBM: {r:.3f}")